In [2]:
# ==================================================================================================
# STEP 02.08 — REPRESENTATION-AWARE CROSS-MODAL TRAIN / VALIDATION SPLIT
# ==================================================================================================
#
# PURPOSE:
#
# Create representation-disjoint training and validation cross-modal pair datasets.
#
# METHODOLOGY:
#
# 1. Load original aligned 32-dimensional representations.
# 2. Split representations into Train and Validation FIRST.
# 3. Construct cross-modal pairs separately inside each partition.
# 4. Create balanced Positive and Negative relationships.
# 5. Verify representation-level independence.
# 6. Save training / validation datasets and metadata.
#
# IMPORTANT:
#
# This replaces the weaker pair-level split methodology from Step 02.07.
#
# Step 02.07:
#     Create pairs → Split pairs
#
# Step 02.08:
#     Split representations → Create pairs independently
#
# ==================================================================================================

print("=" * 100)
print("STEP 02.08 — REPRESENTATION-AWARE CROSS-MODAL SPLIT")
print("=" * 100)

print("""
OBJECTIVE

Create scientifically stronger training and validation datasets by
splitting learned modality representations BEFORE cross-modal pair construction.

CORE PRINCIPLE

A representation used in training must not be reused in validation.

FLOW

Original 32D Representations
        ↓
Representation-Level Train / Validation Split
        ↓
Independent Pair Construction
        ↓
Positive + Negative Relationships
        ↓
Representation-Disjoint Pair Datasets
""")

STEP 02.08 — REPRESENTATION-AWARE CROSS-MODAL SPLIT

OBJECTIVE

Create scientifically stronger training and validation datasets by
splitting learned modality representations BEFORE cross-modal pair construction.

CORE PRINCIPLE

A representation used in training must not be reused in validation.

FLOW

Original 32D Representations
        ↓
Representation-Level Train / Validation Split
        ↓
Independent Pair Construction
        ↓
Positive + Negative Relationships
        ↓
Representation-Disjoint Pair Datasets



In [3]:
# ==================================================================================================
# CELL 2 — IMPORT REQUIRED LIBRARIES
# ==================================================================================================

from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

print("=" * 100)
print("REQUIRED LIBRARIES IMPORTED")
print("=" * 100)

print("NumPy        : Numerical array operations")
print("Pandas       : Metadata handling")
print("Pathlib      : File path management")
print("Scikit-learn : Stratified train / validation splitting")

print("\n✓ ENVIRONMENT READY")

REQUIRED LIBRARIES IMPORTED
NumPy        : Numerical array operations
Pandas       : Metadata handling
Pathlib      : File path management
Scikit-learn : Stratified train / validation splitting

✓ ENVIRONMENT READY


In [4]:
# ==================================================================================================
# CELL 3 — DEFINE PROJECT AND ARTIFACT PATHS
# ==================================================================================================

PROJECT_ROOT = Path(
    r"C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening"
)

# ----------------------------------------------------------------------------------
# ORIGINAL 32D REPRESENTATIONS
# ----------------------------------------------------------------------------------

GENETIC_EMBEDDING_PATH = (
    PROJECT_ROOT /
    "Genetic_Model" /
    "04_cross_modal_alignment" /
    "genetic_32d_aligned_representation.csv"
)

BEHAVIORAL_EMBEDDING_PATH = (
    PROJECT_ROOT /
    "Behavioral_Model" /
    "04_cross_modal_alignment" /
    "behavioral_32d_aligned_representation.csv"
)

ENVIRONMENTAL_EMBEDDING_PATH = (
    PROJECT_ROOT /
    "Environmental_Model" /
    "04_cross_modal_alignment" /
    "environmental_32d_aligned_representation.csv"
)

# ----------------------------------------------------------------------------------
# OUTPUT DIRECTORY
# ----------------------------------------------------------------------------------

OUTPUT_DIR = (
    PROJECT_ROOT /
    "06_cross_modal_learning" /
    "02_cross_modal_pairing" /
    "pairs" /
    "representation_aware_split"
)

print("=" * 100)
print("PROJECT PATH CONFIGURATION")
print("=" * 100)

paths = {
    "Project Root": PROJECT_ROOT,
    "Genetic Embeddings": GENETIC_EMBEDDING_PATH,
    "Behavioral Embeddings": BEHAVIORAL_EMBEDDING_PATH,
    "Environmental Embeddings": ENVIRONMENTAL_EMBEDDING_PATH,
    "Output Directory": OUTPUT_DIR
}

for name, path in paths.items():
    print(f"\n{name}:")
    print(path)

print("\n" + "=" * 100)

for name, path in paths.items():

    if name != "Output Directory":
        print(f"{name:<30} Exists: {path.exists()}")

print("=" * 100)

if not PROJECT_ROOT.exists():
    raise FileNotFoundError("PROJECT_ROOT does not exist.")

if not GENETIC_EMBEDDING_PATH.exists():
    raise FileNotFoundError("Genetic embedding file not found.")

if not BEHAVIORAL_EMBEDDING_PATH.exists():
    raise FileNotFoundError("Behavioral embedding file not found.")

if not ENVIRONMENTAL_EMBEDDING_PATH.exists():
    raise FileNotFoundError("Environmental embedding file not found.")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("\n✓ ALL REQUIRED PATHS VERIFIED")

PROJECT PATH CONFIGURATION

Project Root:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening

Genetic Embeddings:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\Genetic_Model\04_cross_modal_alignment\genetic_32d_aligned_representation.csv

Behavioral Embeddings:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\Behavioral_Model\04_cross_modal_alignment\behavioral_32d_aligned_representation.csv

Environmental Embeddings:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\Environmental_Model\04_cross_modal_alignment\environmental_32d_aligned_representation.csv

Output Directory:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\02_cross_modal_pairing\pairs\representation_aware_split

Project Root                   Exists: True
Genetic Embeddings             Exists: True
Behavioral Embeddings          Exists: True
Environmental Embeddings       Exists: True

✓ ALL R

In [5]:
# ==================================================================================================
# CELL 4 — LOAD ORIGINAL 32D MODALITY REPRESENTATIONS
# ==================================================================================================

print("=" * 100)
print("LOADING ORIGINAL 32D MODALITY REPRESENTATIONS")
print("=" * 100)

# ----------------------------------------------------------------------------------
# LOAD CSV FILES
# ----------------------------------------------------------------------------------

genetic_df = pd.read_csv(GENETIC_EMBEDDING_PATH)

behavioral_df = pd.read_csv(BEHAVIORAL_EMBEDDING_PATH)

environmental_df = pd.read_csv(ENVIRONMENTAL_EMBEDDING_PATH)


# ----------------------------------------------------------------------------------
# EXPLICITLY SELECT EMBEDDING COLUMNS
# ----------------------------------------------------------------------------------

genetic_embedding_columns = [
    col for col in genetic_df.columns
    if col.startswith("Genetic_Aligned_")
]

behavioral_embedding_columns = [
    col for col in behavioral_df.columns
    if col.startswith("Behavioral_32D_")
]

environmental_embedding_columns = [
    col for col in environmental_df.columns
    if col.startswith("Environmental_32D_")
]


# ----------------------------------------------------------------------------------
# EXTRACT EMBEDDINGS
# ----------------------------------------------------------------------------------

genetic_embeddings = (
    genetic_df[genetic_embedding_columns]
    .values
    .astype(np.float32)
)

behavioral_embeddings = (
    behavioral_df[behavioral_embedding_columns]
    .values
    .astype(np.float32)
)

environmental_embeddings = (
    environmental_df[environmental_embedding_columns]
    .values
    .astype(np.float32)
)


# ----------------------------------------------------------------------------------
# EXTRACT LABELS DIRECTLY FROM CSV
# ----------------------------------------------------------------------------------

genetic_labels = genetic_df["Target"].values

behavioral_labels = behavioral_df["Class/ASD"].values

environmental_labels = environmental_df["Target"].values


print("\n" + "=" * 100)
print("REPRESENTATION STRUCTURE")
print("=" * 100)

print(f"""
GENETIC
Embeddings : {genetic_embeddings.shape}
Labels     : {genetic_labels.shape}

BEHAVIORAL
Embeddings : {behavioral_embeddings.shape}
Labels     : {behavioral_labels.shape}

ENVIRONMENTAL
Embeddings : {environmental_embeddings.shape}
Labels     : {environmental_labels.shape}
""")

# ----------------------------------------------------------------------------------
# VALIDATION
# ----------------------------------------------------------------------------------

assert genetic_embeddings.shape[1] == 32
assert behavioral_embeddings.shape[1] == 32
assert environmental_embeddings.shape[1] == 32

assert len(genetic_embeddings) == len(genetic_labels)
assert len(behavioral_embeddings) == len(behavioral_labels)
assert len(environmental_embeddings) == len(environmental_labels)

print("=" * 100)
print("✓ ALL MODALITIES LOADED")
print("✓ EXACTLY 32 EMBEDDING DIMENSIONS VERIFIED")
print("✓ EMBEDDING / LABEL CORRESPONDENCE VERIFIED")
print("=" * 100)

LOADING ORIGINAL 32D MODALITY REPRESENTATIONS

REPRESENTATION STRUCTURE

GENETIC
Embeddings : (116, 32)
Labels     : (116,)

BEHAVIORAL
Embeddings : (880, 32)
Labels     : (880,)

ENVIRONMENTAL
Embeddings : (47768, 32)
Labels     : (47768,)

✓ ALL MODALITIES LOADED
✓ EXACTLY 32 EMBEDDING DIMENSIONS VERIFIED
✓ EMBEDDING / LABEL CORRESPONDENCE VERIFIED


In [6]:
# ==================================================================================================
# CELL 5 — STANDARDIZE DIAGNOSTIC LABELS
# ==================================================================================================

print("=" * 100)
print("STANDARDIZING CLASS LABELS")
print("=" * 100)


def standardize_labels(labels):

    labels = pd.Series(labels).astype(str).str.strip().str.upper()

    mapping = {
        "0": 0,
        "1": 1,
        "NON-ASD": 0,
        "NON_ASD": 0,
        "NO": 0,
        "FALSE": 0,
        "CONTROL": 0,
        "ASD": 1,
        "YES": 1,
        "TRUE": 1
    }

    standardized = labels.map(mapping)

    if standardized.isna().any():

        unknown = labels[standardized.isna()].unique()

        raise ValueError(
            f"Unknown labels detected: {unknown}"
        )

    return standardized.values.astype(int)


genetic_labels = standardize_labels(genetic_labels)

behavioral_labels = standardize_labels(behavioral_labels)

environmental_labels = standardize_labels(environmental_labels)


# ----------------------------------------------------------------------------------
# DISPLAY DISTRIBUTIONS
# ----------------------------------------------------------------------------------

modalities_for_labels = {
    "Genetic": genetic_labels,
    "Behavioral": behavioral_labels,
    "Environmental": environmental_labels
}

for modality_name, labels in modalities_for_labels.items():

    print(f"\n{modality_name}")
    print("-" * 60)

    unique, counts = np.unique(labels, return_counts=True)

    for cls, count in zip(unique, counts):
        print(f"Class {cls}: {count}")

print("\n" + "=" * 100)
print("✓ LABELS STANDARDIZED TO BINARY FORMAT")
print("✓ Class 0 = NON-ASD")
print("✓ Class 1 = ASD")
print("=" * 100)

STANDARDIZING CLASS LABELS

Genetic
------------------------------------------------------------
Class 0: 29
Class 1: 87

Behavioral
------------------------------------------------------------
Class 0: 439
Class 1: 441

Environmental
------------------------------------------------------------
Class 0: 45650
Class 1: 2118

✓ LABELS STANDARDIZED TO BINARY FORMAT
✓ Class 0 = NON-ASD
✓ Class 1 = ASD


In [7]:
# ==================================================================================================
# CELL 6 — DEFINE EXPERIMENT CONFIGURATION
# ==================================================================================================

RANDOM_STATE = 42

TRAIN_RATIO = 0.80
VALIDATION_RATIO = 0.20

np.random.seed(RANDOM_STATE)

print("=" * 100)
print("EXPERIMENT CONFIGURATION")
print("=" * 100)

print(f"Random State      : {RANDOM_STATE}")
print(f"Training Ratio    : {TRAIN_RATIO}")
print(f"Validation Ratio  : {VALIDATION_RATIO}")

print("""
IMPORTANT:

The fixed random seed ensures that the same representation split
and pairing process can be reproduced in future experiments.
""")

print("=" * 100)

EXPERIMENT CONFIGURATION
Random State      : 42
Training Ratio    : 0.8
Validation Ratio  : 0.2

IMPORTANT:

The fixed random seed ensures that the same representation split
and pairing process can be reproduced in future experiments.



In [8]:
# ==================================================================================================
# CELL 7 — REPRESENTATION-LEVEL TRAIN / VALIDATION SPLIT
# ==================================================================================================

print("=" * 100)
print("REPRESENTATION-LEVEL TRAIN / VALIDATION SPLIT")
print("=" * 100)


def split_modality_representations(
    embeddings,
    labels,
    modality_name
):

    indices = np.arange(len(embeddings))

    train_idx, val_idx = train_test_split(
        indices,
        test_size=VALIDATION_RATIO,
        random_state=RANDOM_STATE,
        stratify=labels
    )

    result = {
        "train_embeddings": embeddings[train_idx],
        "train_labels": labels[train_idx],
        "train_indices": train_idx,

        "val_embeddings": embeddings[val_idx],
        "val_labels": labels[val_idx],
        "val_indices": val_idx
    }

    print(f"\n{modality_name}")
    print("-" * 80)

    print("Original representations :", len(embeddings))
    print("Training representations :", len(train_idx))
    print("Validation representations:", len(val_idx))

    return result


# ----------------------------------------------------------------------------------
# SPLIT EACH MODALITY
# ----------------------------------------------------------------------------------

genetic_split = split_modality_representations(
    genetic_embeddings,
    genetic_labels,
    "GENETIC"
)

behavioral_split = split_modality_representations(
    behavioral_embeddings,
    behavioral_labels,
    "BEHAVIORAL"
)

environmental_split = split_modality_representations(
    environmental_embeddings,
    environmental_labels,
    "ENVIRONMENTAL"
)


print("\n" + "=" * 100)
print("✓ REPRESENTATIONS SPLIT BEFORE PAIR CONSTRUCTION")
print("✓ STRATIFIED CLASS DISTRIBUTION PRESERVED")
print("✓ TRAIN / VALIDATION POOLS CREATED")
print("=" * 100)

REPRESENTATION-LEVEL TRAIN / VALIDATION SPLIT

GENETIC
--------------------------------------------------------------------------------
Original representations : 116
Training representations : 92
Validation representations: 24

BEHAVIORAL
--------------------------------------------------------------------------------
Original representations : 880
Training representations : 704
Validation representations: 176

ENVIRONMENTAL
--------------------------------------------------------------------------------
Original representations : 47768
Training representations : 38214
Validation representations: 9554

✓ REPRESENTATIONS SPLIT BEFORE PAIR CONSTRUCTION
✓ STRATIFIED CLASS DISTRIBUTION PRESERVED
✓ TRAIN / VALIDATION POOLS CREATED


In [9]:
# ==================================================================================================
# CELL 8 — VERIFY CLASS DISTRIBUTION AFTER REPRESENTATION SPLIT
# ==================================================================================================

print("=" * 100)
print("CLASS DISTRIBUTION AFTER REPRESENTATION SPLIT")
print("=" * 100)


split_data = {
    "Genetic": genetic_split,
    "Behavioral": behavioral_split,
    "Environmental": environmental_split
}


summary_rows = []

for modality_name, split in split_data.items():

    for partition_name in ["train", "val"]:

        labels = split[f"{partition_name}_labels"]

        class_0 = np.sum(labels == 0)
        class_1 = np.sum(labels == 1)

        summary_rows.append({
            "Modality": modality_name,
            "Partition": partition_name,
            "Class_0_NON_ASD": class_0,
            "Class_1_ASD": class_1,
            "Total": len(labels)
        })


split_summary = pd.DataFrame(summary_rows)

display(split_summary)

print("""
Interpretation:

Each modality has been divided independently into training and
validation representation pools.

Class stratification preserves the diagnostic distribution across
both partitions.
""")

print("=" * 100)
print("✓ SPLIT CLASS DISTRIBUTION VERIFIED")
print("=" * 100)

CLASS DISTRIBUTION AFTER REPRESENTATION SPLIT


,Modality,Partition,Class_0_NON_ASD,Class_1_ASD,Total
0,Genetic,train,23,69,92
1,Genetic,val,6,18,24
2,Behavioral,train,351,353,704
3,Behavioral,val,88,88,176
4,Environmental,train,36520,1694,38214
5,Environmental,val,9130,424,9554



Interpretation:

Each modality has been divided independently into training and
validation representation pools.

Class stratification preserves the diagnostic distribution across
both partitions.

✓ SPLIT CLASS DISTRIBUTION VERIFIED


In [10]:
# ==================================================================================================
# CELL 9 — VERIFY REPRESENTATION-LEVEL TRAIN / VALIDATION DISJOINTNESS
# ==================================================================================================

print("=" * 100)
print("VERIFYING REPRESENTATION DISJOINTNESS")
print("=" * 100)


def check_index_overlap(split, modality_name):

    train_indices = set(split["train_indices"])
    val_indices = set(split["val_indices"])

    overlap = train_indices.intersection(val_indices)

    print(f"\n{modality_name}")
    print("-" * 70)

    print("Train representations     :", len(train_indices))
    print("Validation representations:", len(val_indices))
    print("Overlap                   :", len(overlap))

    assert len(overlap) == 0, (
        f"Representation overlap detected in {modality_name}"
    )


check_index_overlap(genetic_split, "GENETIC")

check_index_overlap(behavioral_split, "BEHAVIORAL")

check_index_overlap(environmental_split, "ENVIRONMENTAL")


print("\n" + "=" * 100)
print("✓ GENETIC TRAIN / VALIDATION OVERLAP = 0")
print("✓ BEHAVIORAL TRAIN / VALIDATION OVERLAP = 0")
print("✓ ENVIRONMENTAL TRAIN / VALIDATION OVERLAP = 0")
print("✓ REPRESENTATION-LEVEL INDEPENDENCE ESTABLISHED")
print("=" * 100)

VERIFYING REPRESENTATION DISJOINTNESS

GENETIC
----------------------------------------------------------------------
Train representations     : 92
Validation representations: 24
Overlap                   : 0

BEHAVIORAL
----------------------------------------------------------------------
Train representations     : 704
Validation representations: 176
Overlap                   : 0

ENVIRONMENTAL
----------------------------------------------------------------------
Train representations     : 38214
Validation representations: 9554
Overlap                   : 0

✓ GENETIC TRAIN / VALIDATION OVERLAP = 0
✓ BEHAVIORAL TRAIN / VALIDATION OVERLAP = 0
✓ ENVIRONMENTAL TRAIN / VALIDATION OVERLAP = 0
✓ REPRESENTATION-LEVEL INDEPENDENCE ESTABLISHED


In [11]:
# ==================================================================================================
# CELL 10 — ORGANIZE TRAIN AND VALIDATION MODALITY POOLS
# ==================================================================================================

print("=" * 100)
print("ORGANIZING MODALITY REPRESENTATION POOLS")
print("=" * 100)


train_modalities = {
    "Genetic": {
        "embeddings": genetic_split["train_embeddings"],
        "labels": genetic_split["train_labels"],
        "original_indices": genetic_split["train_indices"]
    },

    "Behavioral": {
        "embeddings": behavioral_split["train_embeddings"],
        "labels": behavioral_split["train_labels"],
        "original_indices": behavioral_split["train_indices"]
    },

    "Environmental": {
        "embeddings": environmental_split["train_embeddings"],
        "labels": environmental_split["train_labels"],
        "original_indices": environmental_split["train_indices"]
    }
}


validation_modalities = {
    "Genetic": {
        "embeddings": genetic_split["val_embeddings"],
        "labels": genetic_split["val_labels"],
        "original_indices": genetic_split["val_indices"]
    },

    "Behavioral": {
        "embeddings": behavioral_split["val_embeddings"],
        "labels": behavioral_split["val_labels"],
        "original_indices": behavioral_split["val_indices"]
    },

    "Environmental": {
        "embeddings": environmental_split["val_embeddings"],
        "labels": environmental_split["val_labels"],
        "original_indices": environmental_split["val_indices"]
    }
}


for partition_name, modalities in [
    ("TRAINING", train_modalities),
    ("VALIDATION", validation_modalities)
]:

    print(f"\n{partition_name} POOLS")
    print("-" * 80)

    for modality_name, data in modalities.items():

        print(
            f"{modality_name:<20} "
            f"Embeddings: {data['embeddings'].shape}"
        )


print("\n" + "=" * 100)
print("✓ MODALITY POOLS ORGANIZED")
print("=" * 100)

ORGANIZING MODALITY REPRESENTATION POOLS

TRAINING POOLS
--------------------------------------------------------------------------------
Genetic              Embeddings: (92, 32)
Behavioral           Embeddings: (704, 32)
Environmental        Embeddings: (38214, 32)

VALIDATION POOLS
--------------------------------------------------------------------------------
Genetic              Embeddings: (24, 32)
Behavioral           Embeddings: (176, 32)
Environmental        Embeddings: (9554, 32)

✓ MODALITY POOLS ORGANIZED


In [12]:
# ==================================================================================================
# CELL 11 — DEFINE BALANCED CROSS-MODAL PAIRING STRATEGY
# ==================================================================================================

print("=" * 100)
print("CROSS-MODAL PAIRING STRATEGY")
print("=" * 100)

print("""
MODALITY COMBINATIONS

1. Genetic ↔ Behavioral
2. Genetic ↔ Environmental
3. Behavioral ↔ Environmental


POSITIVE PAIR

Same diagnostic class:

Class 0 → Class 0
Class 1 → Class 1

Label = 1


NEGATIVE PAIR

Different diagnostic class:

Class 0 → Class 1
Class 1 → Class 0

Label = 0


PAIRING PRINCIPLE

For each modality combination:

The smaller available class pool determines the pairing budget.

This prevents Cartesian-product explosion and avoids allowing the
largest modality to dominate pair construction.


MATHEMATICAL FORM

For class c:

budget(c) = min(
    number of source representations in class c,
    number of target representations in required class
)

Pairs are sampled one-to-one up to the available minimum budget.
""")

print("=" * 100)

CROSS-MODAL PAIRING STRATEGY

MODALITY COMBINATIONS

1. Genetic ↔ Behavioral
2. Genetic ↔ Environmental
3. Behavioral ↔ Environmental


POSITIVE PAIR

Same diagnostic class:

Class 0 → Class 0
Class 1 → Class 1

Label = 1


NEGATIVE PAIR

Different diagnostic class:

Class 0 → Class 1
Class 1 → Class 0

Label = 0


PAIRING PRINCIPLE

For each modality combination:

The smaller available class pool determines the pairing budget.

This prevents Cartesian-product explosion and avoids allowing the
largest modality to dominate pair construction.


MATHEMATICAL FORM

For class c:

budget(c) = min(
    number of source representations in class c,
    number of target representations in required class
)

Pairs are sampled one-to-one up to the available minimum budget.



In [13]:
# ==================================================================================================
# CELL 12 — DEFINE BALANCED CROSS-MODAL PAIR CONSTRUCTION FUNCTION
# ==================================================================================================

def create_cross_modal_pairs(
    source_data,
    target_data,
    source_modality,
    target_modality,
    random_state=42
):
    """
    Creates balanced positive and negative cross-modal pairs.

    Positive:
        Class 0 -> Class 0
        Class 1 -> Class 1

    Negative:
        Class 0 -> Class 1
        Class 1 -> Class 0

    Pair budget:
        Minimum available representations between the two required
        class pools.

    No Cartesian product is used.
    """

    rng = np.random.default_rng(random_state)

    source_embeddings = source_data["embeddings"]
    source_labels = source_data["labels"]
    source_original_indices = source_data["original_indices"]

    target_embeddings = target_data["embeddings"]
    target_labels = target_data["labels"]
    target_original_indices = target_data["original_indices"]

    pair_sources = []
    pair_targets = []
    pair_labels = []
    metadata_rows = []


    def construct_relationship(
        source_class,
        target_class,
        label,
        pair_type
    ):

        source_idx = np.where(
            source_labels == source_class
        )[0]

        target_idx = np.where(
            target_labels == target_class
        )[0]

        budget = min(
            len(source_idx),
            len(target_idx)
        )

        if budget == 0:
            return

        selected_source = rng.choice(
            source_idx,
            size=budget,
            replace=False
        )

        selected_target = rng.choice(
            target_idx,
            size=budget,
            replace=False
        )

        rng.shuffle(selected_target)

        for src_idx, tgt_idx in zip(
            selected_source,
            selected_target
        ):

            pair_sources.append(
                source_embeddings[src_idx]
            )

            pair_targets.append(
                target_embeddings[tgt_idx]
            )

            pair_labels.append(label)

            metadata_rows.append({
                "source_modality": source_modality,
                "target_modality": target_modality,

                "source_index":
                    int(source_original_indices[src_idx]),

                "target_index":
                    int(target_original_indices[tgt_idx]),

                "source_class":
                    int(source_class),

                "target_class":
                    int(target_class),

                "pair_label":
                    int(label),

                "pair_type":
                    pair_type
            })


    # ------------------------------------------------------------------
    # POSITIVE RELATIONSHIPS
    # ------------------------------------------------------------------

    construct_relationship(
        source_class=0,
        target_class=0,
        label=1,
        pair_type="positive"
    )

    construct_relationship(
        source_class=1,
        target_class=1,
        label=1,
        pair_type="positive"
    )


    # ------------------------------------------------------------------
    # NEGATIVE RELATIONSHIPS
    # ------------------------------------------------------------------

    construct_relationship(
        source_class=0,
        target_class=1,
        label=0,
        pair_type="negative"
    )

    construct_relationship(
        source_class=1,
        target_class=0,
        label=0,
        pair_type="negative"
    )


    return (
        np.asarray(pair_sources, dtype=np.float32),
        np.asarray(pair_targets, dtype=np.float32),
        np.asarray(pair_labels, dtype=np.int64),
        pd.DataFrame(metadata_rows)
    )


print("=" * 100)
print("✓ BALANCED CROSS-MODAL PAIR CONSTRUCTION FUNCTION DEFINED")
print("=" * 100)

✓ BALANCED CROSS-MODAL PAIR CONSTRUCTION FUNCTION DEFINED


In [14]:
# ==================================================================================================
# CELL 13 — DEFINE CROSS-MODAL RELATIONSHIPS
# ==================================================================================================

modality_pairs = [

    ("Genetic", "Behavioral"),

    ("Genetic", "Environmental"),

    ("Behavioral", "Environmental")
]

print("=" * 100)
print("CROSS-MODAL MODALITY RELATIONSHIPS")
print("=" * 100)

for i, (source, target) in enumerate(
    modality_pairs,
    start=1
):
    print(f"{i}. {source} ↔ {target}")

print("\nTotal modality combinations:", len(modality_pairs))

print("=" * 100)

CROSS-MODAL MODALITY RELATIONSHIPS
1. Genetic ↔ Behavioral
2. Genetic ↔ Environmental
3. Behavioral ↔ Environmental

Total modality combinations: 3


In [15]:
# ==================================================================================================
# CELL 14 — CONSTRUCT TRAINING CROSS-MODAL PAIRS
# ==================================================================================================

print("=" * 100)
print("CONSTRUCTING TRAINING CROSS-MODAL PAIRS")
print("=" * 100)

train_sources = []
train_targets = []
train_labels_list = []
train_metadata_list = []

for pair_number, (source_modality, target_modality) in enumerate(
    modality_pairs,
    start=1
):

    print(
        f"\n[{pair_number}/{len(modality_pairs)}] "
        f"{source_modality} ↔ {target_modality}"
    )

    source, target, labels, metadata = create_cross_modal_pairs(
        source_data=train_modalities[source_modality],
        target_data=train_modalities[target_modality],
        source_modality=source_modality,
        target_modality=target_modality,
        random_state=RANDOM_STATE + pair_number
    )

    train_sources.append(source)
    train_targets.append(target)
    train_labels_list.append(labels)
    train_metadata_list.append(metadata)

    print("Pairs created:", len(labels))


# ----------------------------------------------------------------------------------
# COMBINE ALL TRAINING PAIRS
# ----------------------------------------------------------------------------------

X_source_train = np.vstack(train_sources)

X_target_train = np.vstack(train_targets)

y_train = np.concatenate(train_labels_list)

train_metadata = pd.concat(
    train_metadata_list,
    ignore_index=True
)


print("\n" + "=" * 100)
print("FINAL TRAINING PAIR STRUCTURE")
print("=" * 100)

print("Source embeddings :", X_source_train.shape)
print("Target embeddings :", X_target_train.shape)
print("Pair labels       :", y_train.shape)
print("Metadata          :", train_metadata.shape)

print("\nPositive pairs:", np.sum(y_train == 1))
print("Negative pairs:", np.sum(y_train == 0))

print("\n✓ TRAINING PAIRS CONSTRUCTED")

CONSTRUCTING TRAINING CROSS-MODAL PAIRS

[1/3] Genetic ↔ Behavioral
Pairs created: 184

[2/3] Genetic ↔ Environmental
Pairs created: 184

[3/3] Behavioral ↔ Environmental
Pairs created: 1408

FINAL TRAINING PAIR STRUCTURE
Source embeddings : (1776, 32)
Target embeddings : (1776, 32)
Pair labels       : (1776,)
Metadata          : (1776, 8)

Positive pairs: 888
Negative pairs: 888

✓ TRAINING PAIRS CONSTRUCTED


In [16]:
# ==================================================================================================
# CELL 15 — CONSTRUCT VALIDATION CROSS-MODAL PAIRS
# ==================================================================================================

print("=" * 100)
print("CONSTRUCTING VALIDATION CROSS-MODAL PAIRS")
print("=" * 100)

val_sources = []
val_targets = []
val_labels_list = []
val_metadata_list = []

for pair_number, (source_modality, target_modality) in enumerate(
    modality_pairs,
    start=1
):

    print(
        f"\n[{pair_number}/{len(modality_pairs)}] "
        f"{source_modality} ↔ {target_modality}"
    )

    source, target, labels, metadata = create_cross_modal_pairs(
        source_data=validation_modalities[source_modality],
        target_data=validation_modalities[target_modality],
        source_modality=source_modality,
        target_modality=target_modality,
        random_state=RANDOM_STATE + 100 + pair_number
    )

    val_sources.append(source)
    val_targets.append(target)
    val_labels_list.append(labels)
    val_metadata_list.append(metadata)

    print("Pairs created:", len(labels))


# ----------------------------------------------------------------------------------
# COMBINE ALL VALIDATION PAIRS
# ----------------------------------------------------------------------------------

X_source_val = np.vstack(val_sources)

X_target_val = np.vstack(val_targets)

y_val = np.concatenate(val_labels_list)

validation_metadata = pd.concat(
    val_metadata_list,
    ignore_index=True
)


print("\n" + "=" * 100)
print("FINAL VALIDATION PAIR STRUCTURE")
print("=" * 100)

print("Source embeddings :", X_source_val.shape)
print("Target embeddings :", X_target_val.shape)
print("Pair labels       :", y_val.shape)
print("Metadata          :", validation_metadata.shape)

print("\nPositive pairs:", np.sum(y_val == 1))
print("Negative pairs:", np.sum(y_val == 0))

print("\n✓ VALIDATION PAIRS CONSTRUCTED")

CONSTRUCTING VALIDATION CROSS-MODAL PAIRS

[1/3] Genetic ↔ Behavioral
Pairs created: 48

[2/3] Genetic ↔ Environmental
Pairs created: 48

[3/3] Behavioral ↔ Environmental
Pairs created: 352

FINAL VALIDATION PAIR STRUCTURE
Source embeddings : (448, 32)
Target embeddings : (448, 32)
Pair labels       : (448,)
Metadata          : (448, 8)

Positive pairs: 224
Negative pairs: 224

✓ VALIDATION PAIRS CONSTRUCTED


In [17]:
# ==================================================================================================
# CELL 16 — VALIDATE FINAL DATASET STRUCTURE
# ==================================================================================================

print("=" * 100)
print("VALIDATING FINAL DATASET STRUCTURE")
print("=" * 100)


# ----------------------------------------------------------------------------------
# TRAINING
# ----------------------------------------------------------------------------------

assert X_source_train.shape[0] == X_target_train.shape[0]
assert X_source_train.shape[0] == len(y_train)
assert X_source_train.shape[0] == len(train_metadata)

assert X_source_train.shape[1] == 32
assert X_target_train.shape[1] == 32


# ----------------------------------------------------------------------------------
# VALIDATION
# ----------------------------------------------------------------------------------

assert X_source_val.shape[0] == X_target_val.shape[0]
assert X_source_val.shape[0] == len(y_val)
assert X_source_val.shape[0] == len(validation_metadata)

assert X_source_val.shape[1] == 32
assert X_target_val.shape[1] == 32


print(f"""
TRAINING DATASET

Source : {X_source_train.shape}
Target : {X_target_train.shape}
Labels : {y_train.shape}
Metadata: {train_metadata.shape}


VALIDATION DATASET

Source : {X_source_val.shape}
Target : {X_target_val.shape}
Labels : {y_val.shape}
Metadata: {validation_metadata.shape}
""")

print("=" * 100)
print("✓ ALL DIMENSIONS VERIFIED")
print("✓ SOURCE / TARGET / LABEL CORRESPONDENCE VERIFIED")
print("✓ METADATA CORRESPONDENCE VERIFIED")
print("=" * 100)

VALIDATING FINAL DATASET STRUCTURE

TRAINING DATASET

Source : (1776, 32)
Target : (1776, 32)
Labels : (1776,)
Metadata: (1776, 8)


VALIDATION DATASET

Source : (448, 32)
Target : (448, 32)
Labels : (448,)
Metadata: (448, 8)

✓ ALL DIMENSIONS VERIFIED
✓ SOURCE / TARGET / LABEL CORRESPONDENCE VERIFIED
✓ METADATA CORRESPONDENCE VERIFIED


In [18]:
# ==================================================================================================
# CELL 17 — AUDIT PAIR DISTRIBUTION
# ==================================================================================================

print("=" * 100)
print("PAIR DISTRIBUTION AUDIT")
print("=" * 100)


def audit_pair_distribution(
    metadata,
    dataset_name
):

    audit = (
        metadata
        .groupby(
            [
                "source_modality",
                "target_modality",
                "pair_label"
            ]
        )
        .size()
        .reset_index(name="pair_count")
    )

    audit["modality_combination"] = (
        audit["source_modality"]
        + " ↔ "
        + audit["target_modality"]
    )

    print(f"\n{dataset_name}")
    print("-" * 100)

    display(
        audit[
            [
                "modality_combination",
                "pair_label",
                "pair_count"
            ]
        ]
    )

    return audit


train_audit = audit_pair_distribution(
    train_metadata,
    "TRAINING PAIR DISTRIBUTION"
)

validation_audit = audit_pair_distribution(
    validation_metadata,
    "VALIDATION PAIR DISTRIBUTION"
)


print("\n" + "=" * 100)
print("TRAINING TOTAL :", len(train_metadata))
print("VALIDATION TOTAL:", len(validation_metadata))
print("TOTAL PAIRS    :", len(train_metadata) + len(validation_metadata))
print("=" * 100)

PAIR DISTRIBUTION AUDIT

TRAINING PAIR DISTRIBUTION
----------------------------------------------------------------------------------------------------


,modality_combination,pair_label,pair_count
0,Behavioral ↔ Environmental,0,704
1,Behavioral ↔ Environmental,1,704
2,Genetic ↔ Behavioral,0,92
3,Genetic ↔ Behavioral,1,92
4,Genetic ↔ Environmental,0,92
5,Genetic ↔ Environmental,1,92



VALIDATION PAIR DISTRIBUTION
----------------------------------------------------------------------------------------------------


,modality_combination,pair_label,pair_count
0,Behavioral ↔ Environmental,0,176
1,Behavioral ↔ Environmental,1,176
2,Genetic ↔ Behavioral,0,24
3,Genetic ↔ Behavioral,1,24
4,Genetic ↔ Environmental,0,24
5,Genetic ↔ Environmental,1,24



TRAINING TOTAL : 1776
VALIDATION TOTAL: 448
TOTAL PAIRS    : 2224


In [19]:
# ==================================================================================================
# CELL 18 — VALIDATE POSITIVE / NEGATIVE BALANCE
# ==================================================================================================

print("=" * 100)
print("POSITIVE / NEGATIVE BALANCE VALIDATION")
print("=" * 100)


def check_balance(labels, dataset_name):

    positive = np.sum(labels == 1)
    negative = np.sum(labels == 0)

    total = len(labels)

    print(f"\n{dataset_name}")
    print("-" * 70)

    print("Total pairs    :", total)
    print("Positive pairs :", positive)
    print("Negative pairs :", negative)

    print(
        "Positive %     :",
        f"{positive / total * 100:.2f}%"
    )

    print(
        "Negative %     :",
        f"{negative / total * 100:.2f}%"
    )

    print("Balance ratio  :", positive / negative)


check_balance(
    y_train,
    "TRAINING DATASET"
)

check_balance(
    y_val,
    "VALIDATION DATASET"
)


print("\n" + "=" * 100)
print("✓ POSITIVE / NEGATIVE BALANCE VERIFIED")
print("=" * 100)

POSITIVE / NEGATIVE BALANCE VALIDATION

TRAINING DATASET
----------------------------------------------------------------------
Total pairs    : 1776
Positive pairs : 888
Negative pairs : 888
Positive %     : 50.00%
Negative %     : 50.00%
Balance ratio  : 1.0

VALIDATION DATASET
----------------------------------------------------------------------
Total pairs    : 448
Positive pairs : 224
Negative pairs : 224
Positive %     : 50.00%
Negative %     : 50.00%
Balance ratio  : 1.0

✓ POSITIVE / NEGATIVE BALANCE VERIFIED


In [20]:
# ==================================================================================================
# CELL 19 — CHECK EXACT DUPLICATE RELATIONSHIPS
# ==================================================================================================

print("=" * 100)
print("CHECKING EXACT DUPLICATE RELATIONSHIPS")
print("=" * 100)


def check_duplicates(metadata, dataset_name):

    pair_ids = (
        metadata["source_modality"].astype(str)
        + "|"
        + metadata["target_modality"].astype(str)
        + "|"
        + metadata["source_index"].astype(str)
        + "|"
        + metadata["target_index"].astype(str)
        + "|"
        + metadata["pair_label"].astype(str)
    )

    duplicate_count = pair_ids.duplicated().sum()

    print(f"\n{dataset_name}")
    print("-" * 70)

    print("Total relationships :", len(pair_ids))
    print("Unique relationships:", pair_ids.nunique())
    print("Duplicate pairs     :", duplicate_count)

    assert duplicate_count == 0


check_duplicates(
    train_metadata,
    "TRAINING"
)

check_duplicates(
    validation_metadata,
    "VALIDATION"
)


print("\n" + "=" * 100)
print("✓ NO EXACT DUPLICATE PAIRS")
print("=" * 100)

CHECKING EXACT DUPLICATE RELATIONSHIPS

TRAINING
----------------------------------------------------------------------
Total relationships : 1776
Unique relationships: 1776
Duplicate pairs     : 0

VALIDATION
----------------------------------------------------------------------
Total relationships : 448
Unique relationships: 448
Duplicate pairs     : 0

✓ NO EXACT DUPLICATE PAIRS


In [21]:
# ==================================================================================================
# CELL 20 — FINAL TRAIN / VALIDATION REPRESENTATION OVERLAP CHECK
# ==================================================================================================

print("=" * 100)
print("FINAL REPRESENTATION-LEVEL INDEPENDENCE CHECK")
print("=" * 100)


def representation_set(metadata, modality, position):

    if position == "source":

        subset = metadata[
            metadata["source_modality"] == modality
        ]

        return set(
            subset["source_index"].values
        )

    elif position == "target":

        subset = metadata[
            metadata["target_modality"] == modality
        ]

        return set(
            subset["target_index"].values
        )


for modality in [
    "Genetic",
    "Behavioral",
    "Environmental"
]:

    train_ids = set()
    validation_ids = set()

    # Source side
    train_ids.update(
        representation_set(
            train_metadata,
            modality,
            "source"
        )
    )

    validation_ids.update(
        representation_set(
            validation_metadata,
            modality,
            "source"
        )
    )

    # Target side
    train_ids.update(
        representation_set(
            train_metadata,
            modality,
            "target"
        )
    )

    validation_ids.update(
        representation_set(
            validation_metadata,
            modality,
            "target"
        )
    )

    overlap = train_ids.intersection(validation_ids)

    print(f"\n{modality}")
    print("-" * 70)

    print("Training representations :", len(train_ids))
    print("Validation representations:", len(validation_ids))
    print("Overlap                   :", len(overlap))

    assert len(overlap) == 0


print("\n" + "=" * 100)
print("✓ GENETIC OVERLAP = 0")
print("✓ BEHAVIORAL OVERLAP = 0")
print("✓ ENVIRONMENTAL OVERLAP = 0")
print("✓ REPRESENTATION-LEVEL INDEPENDENCE CONFIRMED")
print("=" * 100)

FINAL REPRESENTATION-LEVEL INDEPENDENCE CHECK

Genetic
----------------------------------------------------------------------
Training representations : 92
Validation representations: 24
Overlap                   : 0

Behavioral
----------------------------------------------------------------------
Training representations : 704
Validation representations: 176
Overlap                   : 0

Environmental
----------------------------------------------------------------------
Training representations : 1478
Validation representations: 372
Overlap                   : 0

✓ GENETIC OVERLAP = 0
✓ BEHAVIORAL OVERLAP = 0
✓ ENVIRONMENTAL OVERLAP = 0
✓ REPRESENTATION-LEVEL INDEPENDENCE CONFIRMED


In [22]:
# ==================================================================================================
# CELL 21 — SAVE REPRESENTATION-AWARE TRAIN / VALIDATION DATASETS
# ==================================================================================================

print("=" * 100)
print("SAVING REPRESENTATION-AWARE DATASETS")
print("=" * 100)


# ----------------------------------------------------------------------------------
# DEFINE FILE PATHS
# ----------------------------------------------------------------------------------

TRAIN_PAIR_PATH = (
    OUTPUT_DIR /
    "02_08_train_representation_disjoint_pairs.npz"
)

TRAIN_METADATA_PATH = (
    OUTPUT_DIR /
    "02_08_train_representation_disjoint_metadata.csv"
)

VALIDATION_PAIR_PATH = (
    OUTPUT_DIR /
    "02_08_validation_representation_disjoint_pairs.npz"
)

VALIDATION_METADATA_PATH = (
    OUTPUT_DIR /
    "02_08_validation_representation_disjoint_metadata.csv"
)


# ----------------------------------------------------------------------------------
# SAVE TRAINING DATA
# ----------------------------------------------------------------------------------

np.savez_compressed(
    TRAIN_PAIR_PATH,
    source_embeddings=X_source_train,
    target_embeddings=X_target_train,
    pair_labels=y_train
)

train_metadata.to_csv(
    TRAIN_METADATA_PATH,
    index=False
)


# ----------------------------------------------------------------------------------
# SAVE VALIDATION DATA
# ----------------------------------------------------------------------------------

np.savez_compressed(
    VALIDATION_PAIR_PATH,
    source_embeddings=X_source_val,
    target_embeddings=X_target_val,
    pair_labels=y_val
)

validation_metadata.to_csv(
    VALIDATION_METADATA_PATH,
    index=False
)


print(f"""
TRAINING PAIR DATASET
{TRAIN_PAIR_PATH}

TRAINING METADATA
{TRAIN_METADATA_PATH}

VALIDATION PAIR DATASET
{VALIDATION_PAIR_PATH}

VALIDATION METADATA
{VALIDATION_METADATA_PATH}
""")

print("=" * 100)
print("✓ ALL REPRESENTATION-AWARE ARTIFACTS SAVED")
print("=" * 100)

SAVING REPRESENTATION-AWARE DATASETS

TRAINING PAIR DATASET
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\02_cross_modal_pairing\pairs\representation_aware_split\02_08_train_representation_disjoint_pairs.npz

TRAINING METADATA
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\02_cross_modal_pairing\pairs\representation_aware_split\02_08_train_representation_disjoint_metadata.csv

VALIDATION PAIR DATASET
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\02_cross_modal_pairing\pairs\representation_aware_split\02_08_validation_representation_disjoint_pairs.npz

VALIDATION METADATA
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\02_cross_modal_pairing\pairs\representation_aware_split\02_08_validation_representation_disjoint_metadata.csv

✓ ALL REPRESENTATION-AWARE ARTIFACTS SAVED


In [23]:
# ==================================================================================================
# CELL 22 — VERIFY SAVED ARTIFACTS
# ==================================================================================================

print("=" * 100)
print("VERIFYING SAVED REPRESENTATION-AWARE ARTIFACTS")
print("=" * 100)


# ----------------------------------------------------------------------------------
# FILE EXISTENCE
# ----------------------------------------------------------------------------------

artifacts = {
    "Training Pair Dataset": TRAIN_PAIR_PATH,
    "Training Metadata": TRAIN_METADATA_PATH,
    "Validation Pair Dataset": VALIDATION_PAIR_PATH,
    "Validation Metadata": VALIDATION_METADATA_PATH
}

for name, path in artifacts.items():

    print(f"\n{name}")
    print("-" * 70)

    print("Path  :", path)
    print("Exists:", path.exists())

    assert path.exists()


# ----------------------------------------------------------------------------------
# RELOAD NPZ DATASETS
# ----------------------------------------------------------------------------------

train_saved = np.load(TRAIN_PAIR_PATH)

validation_saved = np.load(VALIDATION_PAIR_PATH)


print("\n" + "=" * 100)
print("RELOADED TRAINING DATASET")
print("=" * 100)

for key in train_saved.files:
    print(f"{key:<25}: {train_saved[key].shape}")


print("\n" + "=" * 100)
print("RELOADED VALIDATION DATASET")
print("=" * 100)

for key in validation_saved.files:
    print(f"{key:<25}: {validation_saved[key].shape}")


# ----------------------------------------------------------------------------------
# RELOAD METADATA
# ----------------------------------------------------------------------------------

train_metadata_saved = pd.read_csv(
    TRAIN_METADATA_PATH
)

validation_metadata_saved = pd.read_csv(
    VALIDATION_METADATA_PATH
)


print("\nTraining metadata rows :", len(train_metadata_saved))
print("Validation metadata rows:", len(validation_metadata_saved))


# ----------------------------------------------------------------------------------
# ROW CORRESPONDENCE
# ----------------------------------------------------------------------------------

assert (
    len(train_saved["pair_labels"])
    == len(train_metadata_saved)
)

assert (
    len(validation_saved["pair_labels"])
    == len(validation_metadata_saved)
)


print("\n" + "=" * 100)
print("✓ ALL FILES EXIST")
print("✓ ALL DATASETS SUCCESSFULLY RELOADED")
print("✓ DATASET / METADATA CORRESPONDENCE VERIFIED")
print("=" * 100)

VERIFYING SAVED REPRESENTATION-AWARE ARTIFACTS

Training Pair Dataset
----------------------------------------------------------------------
Path  : C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\02_cross_modal_pairing\pairs\representation_aware_split\02_08_train_representation_disjoint_pairs.npz
Exists: True

Training Metadata
----------------------------------------------------------------------
Path  : C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\02_cross_modal_pairing\pairs\representation_aware_split\02_08_train_representation_disjoint_metadata.csv
Exists: True

Validation Pair Dataset
----------------------------------------------------------------------
Path  : C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\02_cross_modal_pairing\pairs\representation_aware_split\02_08_validation_representation_disjoint_pairs.npz
Exists: True

Validation Me

In [24]:
# ==================================================================================================
# CELL 23 — FINAL PAIR COUNT ORIGIN AUDIT
# ==================================================================================================

print("=" * 100)
print("STEP 02.08 — FINAL PAIR COUNT ORIGIN")
print("=" * 100)


def pair_origin_audit(metadata, dataset_name):

    print(f"\n{dataset_name}")
    print("=" * 100)

    audit = (
        metadata
        .groupby(
            [
                "source_modality",
                "target_modality",
                "source_class",
                "target_class",
                "pair_label"
            ]
        )
        .size()
        .reset_index(name="pair_count")
    )

    display(audit)

    print("\nTOTAL PAIRS:", audit["pair_count"].sum())

    return audit


train_origin_audit = pair_origin_audit(
    train_metadata,
    "TRAINING PAIR ORIGIN"
)

validation_origin_audit = pair_origin_audit(
    validation_metadata,
    "VALIDATION PAIR ORIGIN"
)


print("\n" + "=" * 100)
print("FINAL TOTAL")
print("=" * 100)

print("Training pairs   :", len(y_train))
print("Validation pairs :", len(y_val))
print("Total pairs      :", len(y_train) + len(y_val))

print("=" * 100)
print("✓ PAIR COUNT ORIGIN FULLY DOCUMENTED")
print("=" * 100)

STEP 02.08 — FINAL PAIR COUNT ORIGIN

TRAINING PAIR ORIGIN


,source_modality,target_modality,source_class,target_class,pair_label,pair_count
0,Behavioral,Environmental,0,0,1,351
1,Behavioral,Environmental,0,1,0,351
2,Behavioral,Environmental,1,0,0,353
3,Behavioral,Environmental,1,1,1,353
4,Genetic,Behavioral,0,0,1,23
5,Genetic,Behavioral,0,1,0,23
6,Genetic,Behavioral,1,0,0,69
7,Genetic,Behavioral,1,1,1,69
8,Genetic,Environmental,0,0,1,23
9,Genetic,Environmental,0,1,0,23



TOTAL PAIRS: 1776

VALIDATION PAIR ORIGIN


,source_modality,target_modality,source_class,target_class,pair_label,pair_count
0,Behavioral,Environmental,0,0,1,88
1,Behavioral,Environmental,0,1,0,88
2,Behavioral,Environmental,1,0,0,88
3,Behavioral,Environmental,1,1,1,88
4,Genetic,Behavioral,0,0,1,6
5,Genetic,Behavioral,0,1,0,6
6,Genetic,Behavioral,1,0,0,18
7,Genetic,Behavioral,1,1,1,18
8,Genetic,Environmental,0,0,1,6
9,Genetic,Environmental,0,1,0,6



TOTAL PAIRS: 448

FINAL TOTAL
Training pairs   : 1776
Validation pairs : 448
Total pairs      : 2224
✓ PAIR COUNT ORIGIN FULLY DOCUMENTED


In [25]:
# ==================================================================================================
# CELL 24 — STEP 02.08 FINAL SUMMARY
# ==================================================================================================

print("=" * 100)
print("STEP 02.08 — REPRESENTATION-AWARE SPLIT FINAL SUMMARY")
print("=" * 100)

print(f"""
INPUT REPRESENTATIONS
----------------------------------------------------------------------------------------------------

Genetic
Original representations : {len(genetic_embeddings)}
Dimension               : {genetic_embeddings.shape[1]}

Behavioral
Original representations : {len(behavioral_embeddings)}
Dimension               : {behavioral_embeddings.shape[1]}

Environmental
Original representations : {len(environmental_embeddings)}
Dimension               : {environmental_embeddings.shape[1]}


METHODOLOGY
----------------------------------------------------------------------------------------------------

1. Original learned 32D representations were loaded.

2. Each modality was independently split into:

   • Training representations
   • Validation representations

3. Stratified splitting preserved diagnostic class distributions.

4. Train and validation representations were kept disjoint.

5. Cross-modal pairs were constructed independently inside:

   • Training representation pools
   • Validation representation pools

6. Positive relationships:

   Class 0 → Class 0
   Class 1 → Class 1

7. Negative relationships:

   Class 0 → Class 1
   Class 1 → Class 0

8. Pair budgets were determined using the minimum available
   class-specific representation pool.

9. Cartesian-product pairing was avoided.

10. All relationships were stored with complete metadata.


FINAL DATASETS
----------------------------------------------------------------------------------------------------

TRAINING

Total pairs       : {len(y_train)}
Positive pairs    : {np.sum(y_train == 1)}
Negative pairs    : {np.sum(y_train == 0)}

Source shape      : {X_source_train.shape}
Target shape      : {X_target_train.shape}

Metadata rows     : {len(train_metadata)}


VALIDATION

Total pairs       : {len(y_val)}
Positive pairs    : {np.sum(y_val == 1)}
Negative pairs    : {np.sum(y_val == 0)}

Source shape      : {X_source_val.shape}
Target shape      : {X_target_val.shape}

Metadata rows     : {len(validation_metadata)}


REPRESENTATION INDEPENDENCE
----------------------------------------------------------------------------------------------------

✓ Genetic train / validation overlap       = 0
✓ Behavioral train / validation overlap    = 0
✓ Environmental train / validation overlap = 0


SCIENTIFIC IMPROVEMENT OVER STEP 02.07
----------------------------------------------------------------------------------------------------

OLD APPROACH:

Create pairs
      ↓
Split pairs

Potential issue:
Individual representations could appear in both partitions.


NEW APPROACH:

Split representations
      ↓
Create pairs independently

Result:

✓ Representation-level independence
✓ No train / validation representation reuse
✓ No exact duplicate relationships
✓ Balanced positive / negative relationships
✓ Original 32D representations preserved
✓ Full pair metadata preserved
✓ Stronger validation methodology


SAVED OUTPUTS
----------------------------------------------------------------------------------------------------

Training Pair Dataset:
{TRAIN_PAIR_PATH.name}

Training Metadata:
{TRAIN_METADATA_PATH.name}

Validation Pair Dataset:
{VALIDATION_PAIR_PATH.name}

Validation Metadata:
{VALIDATION_METADATA_PATH.name}


NEXT STEP
----------------------------------------------------------------------------------------------------

PHASE 03 — CROSS-MODAL TRAINING

Use:

Training:
02_08_train_representation_disjoint_pairs.npz

Validation:
02_08_validation_representation_disjoint_pairs.npz
""")

print("=" * 100)
print("✓ STEP 02.08 COMPLETED SUCCESSFULLY")
print("✓ REPRESENTATION-AWARE DATASETS READY FOR PHASE 03")
print("=" * 100)

STEP 02.08 — REPRESENTATION-AWARE SPLIT FINAL SUMMARY

INPUT REPRESENTATIONS
----------------------------------------------------------------------------------------------------

Genetic
Original representations : 116
Dimension               : 32

Behavioral
Original representations : 880
Dimension               : 32

Environmental
Original representations : 47768
Dimension               : 32


METHODOLOGY
----------------------------------------------------------------------------------------------------

1. Original learned 32D representations were loaded.

2. Each modality was independently split into:

   • Training representations
   • Validation representations

3. Stratified splitting preserved diagnostic class distributions.

4. Train and validation representations were kept disjoint.

5. Cross-modal pairs were constructed independently inside:

   • Training representation pools
   • Validation representation pools

6. Positive relationships:

   Class 0 → Class 0
   Class 1 →